In [1]:
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
import textwrap
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import xarray as xr
import glob

In [2]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)

In [4]:
# combine mslp files (from NOAA https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis2/Dailies/surface/)
ds = xr.open_mfdataset(
    "datasets/MSLP/NOAA/*.nc",
    combine = "by_coords",
    preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore")
)

# select the MSLP variable
mslp = ds["mslp"]

In [6]:
# convert lon to -180-180
mslp = mslp.assign_coords(
    lon=(((mslp.lon + 180) % 360) - 180)
).sortby("lon")

# add CRS and spatial dims
mslp = mslp.rio.write_crs("EPSG:4326")
mslp = mslp.rio.set_spatial_dims(x_dim="lon", y_dim="lat")

# filter to N Atlantic basin
region = basins[basins["basin name"] == "N Atlantic"]

# filter to hurricane season
mslp_full = (
    mslp
    .where(mslp.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
    .rio.clip(region.geometry, region.crs, drop=True)
)

# aggregate daily means to monthly means
mslp_monthly = mslp_full.resample(time="MS").mean("time")

print(mslp_monthly)

<xarray.DataArray 'mslp' (time: 557, lat: 37, lon: 73)> Size: 6MB
dask.array<where, shape=(557, 37, 73), dtype=float32, chunksize=(1, 37, 73), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 4kB 1979-06-01 1979-07-01 ... 2025-10-01
  * lat          (lat) float32 148B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float32 292B -160.0 -157.5 -155.0 ... 15.0 17.5 20.0
    spatial_ref  int64 8B 0
Attributes: (12/14)
    long_name:      Daily Mean Sea Level Pressure
    units:          Pascals
    precision:      0
    GRIB_id:        2
    GRIB_name:      PRMSL
    var_desc:       Mean Sea Level Pressure
    ...             ...
    statistic:      Mean
    parent_stat:    Individual Obs
    standard_name:  pressure
    cell_methods:   time: mean (of 4 6-hourly values in one day)
    actual_range:   [ 93682. 106592.]
    valid_range:    [ 77000. 125000.]


In [7]:
# save filtered datasets
mslp_monthly.to_netcdf("datasets/MSLP/post-processing/MSLP_monthlyMean_1979-2025.nc")
